In [1]:
# -*- coding: utf-8 -*-
"""
OFFSET QUANTIFICATION & FIGURE: CTD vs MVP (Leg 1)
- Match espacial + temporal (CTD más próximo a cada MVP).
- Cuantificación del offset instrumental por capas de profundidad.
- Figura de 3 paneles (T, C, S): CTD (negro), MVP Raw (gris), MVP Smooth (rojo).
- Tabla resumen de métricas (RMSD, offset por capa).
"""

import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
from pathlib import Path
from geopy.distance import geodesic
import gsw
import re
import io
import warnings

warnings.filterwarnings("ignore")

# ==========================================
# 1. CONFIGURACIÓN
# ==========================================
BASE_ROOT = Path(r"C:\Users\ASUS\Desktop\MVP\MVP_fastswot_nc_final_processing")

DIR_MVP_RAW  = BASE_ROOT / "Leg1" / "processed_step1_highres_qc"
DIR_MVP_SMOOTH = BASE_ROOT / "Leg1" / "processed_step4_loess"
DIR_CTD = Path(r"C:/Users/ASUS/OneDrive - Universitat de les Illes Balears/SWOT/CTD/CTD_data/leg1/HM/")

OUTDIR = BASE_ROOT / "FIGURES_OFFSET_DIAGNOSIS"
OUTDIR.mkdir(parents=True, exist_ok=True)

# Parámetros de match y análisis
MAX_DIST_KM  = 4.0       # Distancia máxima CTD-MVP (km)
MAX_TIME_MIN = 180.0      # Diferencia temporal máxima (min)
Z_MAX_PLOT   = 100.0      # Profundidad máxima para gráficos y métricas (dbar)
SPLIT_DEPTH  = 60.0       # Separación de capas para offset (dbar)
CTD_SMOOTH_WIN = 8        # Ventana de suavizado para CTD (rolling)

# ==========================================
# 2. FUNCIONES AUXILIARES
# ==========================================

def read_ctd_cnv(path):
    """Lee un archivo .cnv de CTD (SeaBird). Devuelve dict con p, t, s, c, lat, lon, time."""
    try:
        with open(path, "r", encoding="latin-1") as f:
            lines = f.readlines()

        lat, lon, time_val, start_idx = np.nan, np.nan, pd.NaT, 0
        col_map = {}

        for i, line in enumerate(lines):
            if "NMEA Latitude" in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p) >= 2:
                    lat = float(p[0]) + float(p[1]) / 60 * (-1 if "S" in line else 1)
            if "NMEA Longitude" in line:
                p = re.findall(r"[-+]?\d*\.\d+|\d+", line)
                if len(p) >= 2:
                    lon = float(p[0]) + float(p[1]) / 60 * (-1 if "W" in line else 1)
            if "start_time" in line:
                try:
                    time_val = pd.to_datetime(line.split("=")[1].strip().split("[")[0])
                except Exception:
                    pass
            if "# name" in line:
                idx = int(re.search(r"name (\d+)", line).group(1))
                name = line.split("=")[1].split(":")[0].strip().lower()
                if name in ("prdm", "pres", "pressure"):
                    col_map["p"] = idx
                elif name in ("t090c", "t190c", "temp", "tv290c"):
                    col_map["t"] = idx
                elif name in ("sal00", "sal11", "sal", "psal"):
                    col_map["s"] = idx
                elif name in ("c0s/m", "c1s/m", "cond", "c0ms/cm"):
                    col_map["c"] = idx
            if "*END*" in line:
                start_idx = i + 1
                break

        df = pd.read_csv(
            io.StringIO("".join(lines[start_idx:])),
            delim_whitespace=True,
            header=None,
            on_bad_lines="skip",
        )

        p = df.iloc[:, col_map["p"]].values
        t = df.iloc[:, col_map["t"]].values
        s = df.iloc[:, col_map["s"]].values
        c = df.iloc[:, col_map["c"]].values if "c" in col_map else None
        # Conversión S/m → mS/cm si es necesario
        if c is not None and np.nanmean(c) < 10:
            c = c * 10.0

        return {"lat": lat, "lon": lon, "time": time_val, "id": path.stem,
                "p": p, "t": t, "s": s, "c": c}
    except Exception:
        return None


def load_mvp_catalog(mvp_dir):
    """Construye un DataFrame con metadatos de todos los perfiles MVP (.nc)."""
    data = []
    for f in sorted(mvp_dir.glob("*.nc")):
        try:
            with xr.open_dataset(f) as ds:
                lat, lon = np.nan, np.nan
                for ln in ("latitude", "lat"):
                    if ln in ds:
                        lat = float(ds[ln].values.flatten()[0])
                        break
                for ln in ("longitude", "lon"):
                    if ln in ds:
                        lon = float(ds[ln].values.flatten()[0])
                        break
                t_val = (
                    pd.to_datetime(ds.time.values.flatten()[0])
                    if "time" in ds
                    else pd.NaT
                )
                if np.isfinite(lat) and np.isfinite(lon):
                    data.append(
                        {"file": f.name, "lat": lat, "lon": lon, "time": t_val, "path": f}
                    )
        except Exception:
            pass
    return pd.DataFrame(data)


def find_closest_mvp(ctd, df_mvp):
    """Devuelve el MVP más cercano que cumple los filtros de distancia y tiempo."""
    if df_mvp.empty or not (np.isfinite(ctd["lat"]) and np.isfinite(ctd["lon"])):
        return None, np.nan, np.nan

    dists = np.array(
        [geodesic((ctd["lat"], ctd["lon"]), (r.lat, r.lon)).km for r in df_mvp.itertuples()]
    )
    best_idx = int(np.argmin(dists))
    best_row = df_mvp.iloc[best_idx]
    dist_km = dists[best_idx]

    dt_min = (
        abs((best_row["time"] - ctd["time"]).total_seconds() / 60.0)
        if pd.notna(best_row["time"]) and pd.notna(ctd["time"])
        else 999.0
    )

    if dist_km <= MAX_DIST_KM and dt_min <= MAX_TIME_MIN:
        return best_row, dist_km, dt_min
    return None, dist_km, dt_min


# ==========================================
# 3. PROCESAMIENTO: MATCHING + OFFSET
# ==========================================

print("Cargando catálogo MVP y estaciones CTD...")
df_mvp = load_mvp_catalog(DIR_MVP_RAW)
ctd_files = sorted(DIR_CTD.glob("d*.cnv"))

if df_mvp.empty:
    raise RuntimeError("Catálogo MVP vacío – revisa la ruta DIR_MVP_RAW.")

results = []

for ctd_f in ctd_files:
    ctd = read_ctd_cnv(ctd_f)
    if ctd is None:
        continue

    best_mvp, dist_km, dt_min = find_closest_mvp(ctd, df_mvp)
    if best_mvp is None:
        print(f"  ⏩ {ctd['id']}: sin match válido (dist={dist_km:.2f} km, dt={dt_min:.0f} min)")
        continue

    # Archivos correspondientes (raw → step4_loess)
    f_smooth = DIR_MVP_SMOOTH / best_mvp["file"].replace("_step1_qc.nc", "_step4_loess.nc")
    if not f_smooth.exists():
        print(f"  ⏩ {ctd['id']}: falta archivo smooth ({f_smooth.name})")
        continue

    try:
        # --- Lectura MVP raw + smooth ---
        with xr.open_dataset(best_mvp["path"]) as ds_raw, xr.open_dataset(f_smooth) as ds_sm:
            p_mvp   = ds_raw.pressure.values
            t_raw   = ds_raw.t1.values
            c_raw   = ds_raw.c1.values
            if np.nanmean(c_raw) < 10:
                c_raw = c_raw * 10  # S/m → mS/cm
            s_raw   = gsw.SP_from_C(c_raw, t_raw, p_mvp)

            t_smooth = ds_sm.t1_smooth.values
            s_smooth = ds_sm.salinity_smooth.values   # SIN offset aplicado
            c_smooth = gsw.C_from_SP(s_smooth, t_smooth, p_mvp)

        # --- Suavizado CTD para comparación visual ---
        t_ctd_sm = pd.Series(ctd["t"]).rolling(CTD_SMOOTH_WIN, center=True, min_periods=1).mean().values
        s_ctd_sm = pd.Series(ctd["s"]).rolling(CTD_SMOOTH_WIN, center=True, min_periods=1).mean().values

        # --- Grilla común para métricas ---
        p_grid  = np.arange(10, Z_MAX_PLOT, 1.0)
        s_ctd_i = np.interp(p_grid, ctd["p"], ctd["s"])
        s_raw_i = np.interp(p_grid, p_mvp, s_raw)
        s_sm_i  = np.interp(p_grid, p_mvp, s_smooth)

        mask_all = np.isfinite(s_ctd_i) & np.isfinite(s_raw_i) & np.isfinite(s_sm_i)
        if mask_all.sum() < 10:
            continue

        # Offset por capas (CTD - MVP_smooth)
        mask_shallow = mask_all & (p_grid <= SPLIT_DEPTH)
        mask_deep    = mask_all & (p_grid > SPLIT_DEPTH)

        off_shallow = np.nanmedian(s_ctd_i[mask_shallow] - s_sm_i[mask_shallow]) if mask_shallow.sum() > 5 else np.nan
        off_deep    = np.nanmedian(s_ctd_i[mask_deep]    - s_sm_i[mask_deep])    if mask_deep.sum() > 5    else np.nan
        off_global  = np.nanmedian(s_ctd_i[mask_all]     - s_sm_i[mask_all])

        # RMSD
        rmsd_raw = np.sqrt(np.nanmean((s_ctd_i[mask_all] - s_raw_i[mask_all]) ** 2))
        rmsd_sm  = np.sqrt(np.nanmean((s_ctd_i[mask_all] - s_sm_i[mask_all]) ** 2))
        improv   = (1 - rmsd_sm / rmsd_raw) * 100 if rmsd_raw > 0 else 0.0

        results.append({
            "CTD_Station": ctd["id"],
            "MVP_File": best_mvp["file"],
            "Dist_km": round(dist_km, 2),
            "Time_min": round(dt_min, 1),
            "Offset_10-60m": round(off_shallow, 4),
            "Offset_>60m": round(off_deep, 4),
            "Offset_Global": round(off_global, 4),
            "RMSD_Raw": round(rmsd_raw, 4),
            "RMSD_Smooth": round(rmsd_sm, 4),
            "Improvement_%": round(improv, 1),
        })

        # --- FIGURA 3 PANELES ---
        fig, axs = plt.subplots(1, 3, figsize=(16, 7), sharey=True, constrained_layout=True)

        # (a) Temperatura
        axs[0].plot(t_raw, p_mvp, color="gray", lw=0.6, alpha=0.5, label="MVP Raw")
        axs[0].plot(t_smooth, p_mvp, "r", lw=1.5, label="MVP Smooth")
        axs[0].plot(t_ctd_sm, ctd["p"], "k", lw=2, label="CTD")
        axs[0].set_xlabel("Temperature (°C)")
        axs[0].set_ylabel("Pressure (dbar)")
        axs[0].set_title("(a) Temperature", fontweight="bold", loc="left")
        axs[0].legend(loc="lower right", fontsize=8)

        # (b) Conductividad
        axs[1].plot(c_raw, p_mvp, color="gray", lw=0.6, alpha=0.5, label="MVP Raw")
        axs[1].plot(c_smooth, p_mvp, "r", lw=1.5, label="MVP Smooth")
        if ctd["c"] is not None:
            c_ctd_sm = pd.Series(ctd["c"]).rolling(CTD_SMOOTH_WIN, center=True, min_periods=1).mean().values
            axs[1].plot(c_ctd_sm, ctd["p"], "k", lw=2, label="CTD")
        axs[1].set_xlabel("Conductivity (mS/cm)")
        axs[1].set_title("(b) Conductivity", fontweight="bold", loc="left")

        # (c) Salinidad
        axs[2].plot(s_raw, p_mvp, color="gray", lw=0.6, alpha=0.5, label="MVP Raw")
        axs[2].plot(s_smooth, p_mvp, "r", lw=1.8, label="MVP Smooth")
        axs[2].plot(s_ctd_sm, ctd["p"], "k", lw=2.5, label="CTD")
        s_mid = np.nanmedian(s_smooth)
        axs[2].set_xlim(s_mid - 0.4, s_mid + 0.4)
        axs[2].set_xlabel("Salinity (PSU)")
        axs[2].set_title("(c) Salinity", fontweight="bold", loc="left")
        axs[2].legend(loc="lower right", fontsize=8)

        for ax in axs:
            ax.invert_yaxis()
            ax.set_ylim(Z_MAX_PLOT, 0)
            ax.grid(True, alpha=0.2)

        # Cuadro de texto con métricas
        txt = (
            f"CTD: {ctd['id']}\n"
            f"Dist: {dist_km:.2f} km | Δt: {dt_min:.0f} min\n"
            f"Offset (10-{int(SPLIT_DEPTH)}m): {off_shallow:+.4f} PSU\n"
            f"Offset (>{int(SPLIT_DEPTH)}m): {off_deep:+.4f} PSU\n"
            f"RMSD smooth: {rmsd_sm:.4f}"
        )
        axs[0].text(
            0.05, 0.05, txt, transform=axs[0].transAxes, fontsize=9,
            verticalalignment="bottom",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.9),
        )

        fig.savefig(OUTDIR / f"Offset_Diagnosis_{ctd['id']}.png", dpi=300)
        plt.close(fig)
        print(f"  ✅ {ctd['id']} ↔ {best_mvp['file'][:20]}  (d={dist_km:.2f} km, Δt={dt_min:.0f} min)")

    except Exception as e:
        print(f"  ❌ Error {ctd['id']}: {e}")

# ==========================================
# 4. TABLA RESUMEN Y PROMEDIOS
# ==========================================

if results:
    df_res = pd.DataFrame(results)
    print("\n" + "=" * 90)
    print(f"{'OFFSET INSTRUMENTAL – RESUMEN (CTD - MVP Smooth)':^90}")
    print("=" * 90)
    print(df_res.to_string(index=False))
    print("-" * 90)
    print(f"  Offset medio (10-{int(SPLIT_DEPTH)}m) : {df_res['Offset_10-60m'].mean():.4f} PSU")
    print(f"  Offset medio (>{int(SPLIT_DEPTH)}m)   : {df_res['Offset_>60m'].mean():.4f} PSU")
    print(f"  Offset medio (global)   : {df_res['Offset_Global'].mean():.4f} PSU")
    print(f"  RMSD medio (raw)        : {df_res['RMSD_Raw'].mean():.4f}")
    print(f"  RMSD medio (smooth)     : {df_res['RMSD_Smooth'].mean():.4f}")
    print(f"  Mejora media            : {df_res['Improvement_%'].mean():.1f} %")
    print("=" * 90)
    csv_path = OUTDIR / "Offset_Metrics_Leg1.csv"
    df_res.to_csv(csv_path, index=False)
    print(f"\nTabla guardada en: {csv_path}")
    print(f"Figuras guardadas en: {OUTDIR}")
else:
    print("\n⚠️ No se encontraron coincidencias válidas.")

c:\Users\ASUS\anaconda3\envs\env_elisabet\lib\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


Cargando catálogo MVP y estaciones CTD...
  ✅ dstation_01 ↔ imedea-socib_fast-sw  (d=0.75 km, Δt=47 min)
  ✅ dstation_02 ↔ imedea-socib_fast-sw  (d=0.48 km, Δt=24 min)
  ✅ dstation_03 ↔ imedea-socib_fast-sw  (d=0.59 km, Δt=4 min)
  ✅ dstation_04 ↔ imedea-socib_fast-sw  (d=0.39 km, Δt=34 min)
  ✅ dstation_05 ↔ imedea-socib_fast-sw  (d=0.95 km, Δt=34 min)
  ⏩ dstation_06: sin match válido (dist=0.52 km, dt=540 min)
  ⏩ dstation_07: sin match válido (dist=0.14 km, dt=514 min)
  ⏩ dstation_08: sin match válido (dist=0.35 km, dt=532 min)
  ✅ dstation_09 ↔ imedea-socib_fast-sw  (d=0.46 km, Δt=38 min)
  ✅ dstation_10 ↔ imedea-socib_fast-sw  (d=0.23 km, Δt=41 min)
  ✅ dstation_11 ↔ imedea-socib_fast-sw  (d=0.24 km, Δt=35 min)
  ✅ dstation_12 ↔ imedea-socib_fast-sw  (d=0.36 km, Δt=63 min)

                     OFFSET INSTRUMENTAL – RESUMEN (CTD - MVP Smooth)                     
CTD_Station                                     MVP_File  Dist_km  Time_min  Offset_10-60m  Offset_>60m  Offset_Globa